In [1]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

#import cv2


In [2]:
# import easyocr

# # 1. Create a reader object
# # Specify the languages you want to read. English ('en') is compatible with all.
# # Set 'gpu=False' if you do not have a CUDA-capable GPU.
# reader = easyocr.Reader(['en']) # Example for English text

# # 2. Load an image and perform OCR
# # You can pass a file path, an OpenCV image object (numpy array), or an image URL.
# result = reader.readtext("./output/5.5_header_rotated.jpg")


In [3]:
# result

In [4]:
# from src.config import TrackerConfig
# from src.scanner import HabitTrackerScanner

# # Configuration
# config = TrackerConfig(
#     output_dir="./output"
# )

# # Initialize scanner
# scanner = HabitTrackerScanner(config)

# # Scan the sample image
# image_path = "./input/version_2_filled2.jpg"
# scanner.scan(image_path)


In [5]:
# scanner.column_names

In [6]:
#from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image

# Load once, use offline afterwards
#processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
#model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

In [7]:
from PIL import Image
import os

def crop_list_into_items(image_path, num_items=15):
    """
    Crop a list image into equally-spaced items.
    
    Args:
        image_path: Path to the input image
        num_items: Number of items to extract (default: 15)
        output_dir: Directory to save cropped images
    
    Returns:
        List of cropped PIL Image objects
    """
    # Load the image
    image = Image.open(image_path)
    width, height = image.size
    
    print(f"Image dimensions: {width}x{height}")
    
    # Calculate the height of each item
    item_height = height / num_items
    
    print(f"Each item height: {item_height:.2f} pixels")
    
    # Crop each item
    cropped_images = []
    
    for i in range(num_items):
        # Calculate crop box (left, top, right, bottom)
        top = int(i * item_height)
        bottom = int((i + 1) * item_height)
        
        # Crop the image
        crop_box = (0, top, width, bottom)
        cropped_img = image.crop(crop_box)
        
        cropped_images.append(cropped_img)
    
    return cropped_images

# Crop into 15 items
cropped_images = crop_list_into_items(
    image_path="output/5.5_header_rotated.jpg",
    num_items=15,
)


Image dimensions: 250x840
Each item height: 56.00 pixels


In [8]:
# for i, image in enumerate(cropped_images):
#     print(f"Habit {i}: ", end="")
#     pixel_values = processor(image, return_tensors="pt").pixel_values
#     generated_ids = model.generate(pixel_values)
#     text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
#     print(text)
    
#     if i == 5:
#         break

In [9]:
image = cropped_images[0]

In [ ]:
from paddleocr import PaddleOCR

/Users/trung/opt/anaconda3/envs/habits/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [ ]:

ocr = PaddleOCR(use_textline_orientation=True, lang='en')


In [ ]:
# Convert PIL Image to numpy array (PaddleOCR expects numpy arrays)
img_array = np.array(image)

# Run OCR on numpy array
ocr_result = ocr.ocr(img_array, cls=True)

# Extract text from results
text_lines = []
if ocr_result and ocr_result[0]:
    for line in ocr_result[0]:
        text = line[1][0]  # line[1][0] is the text, line[1][1] is confidence
        confidence = line[1][1]
        text_lines.append({
            'text': text,
            'confidence': confidence
        })
